# What is PySpark?

PySpark is a Python API for Apache Spark used to:

- Process large datasets
- Transform data
- Analyze data

In simple terms: **PySpark helps us work with big data efficiently**

---

# What is CRUD?

CRUD stands for:

| Operation | Meaning | PySpark Equivalent |
| --- | --- | --- |
| C | Create | Create DataFrame / Add rows |
| R | Read | Select / Filter |
| U | Update | withColumn |
| D | Delete | Filter (remove rows) |

---

# 1. CREATE → Add Data

### Purpose:

Create or add data into a DataFrame

In [0]:
data=[
    (1,'John Smith','UK','2019-01-01'),
    (2, 'Alice Brown', 'USA', '2024-02-15'),
    (3, 'Raj Patel', 'India', '2024-03-01')
]
columns=['customer_id','name','country','signup_date']

df=spark.createDataFrame(data,columns)
df.display()

In [0]:
#A PySpark DataFrame is immutable, so you don't directly insert a row into df. Instead, create another DataFrame for the new row and union it with the existing one.

new_data=[
    (4, 'Emma Wilson', 'UK', '2024-04-01')
]
new_df=spark.createDataFrame(new_data,columns)
new_df.display()

In [0]:
df=df.union(new_df)
df.display()

## Key Points:

- PySpark DataFrames are **immutable**
- You don’t modify → you create a new DataFrame

## Practice:

- Add 2 new customers, from different countries

In [0]:
new_data=[
    (5,'Jane Doe','Australia','2024-05-01'),
    (6, 'Bob Johnson', 'Canada','2024-06-01')
]
new_df=spark.createDataFrame(new_data,columns)
df=df.union(new_df)
df.display()

# 2. READ → View Data

### Purpose:

Retrieve data

In [0]:
#df.show()
df.select('*').display()

In [0]:
df.filter(df.country=='UK').display()

## Key Points:

- `.show() and .display()` displays data
- `.select()` chooses columns
- `.filter()` applies conditions

## Practice:

- Show all customers
- Show only UK customers
- Show only names

In [0]:
#df.select('*').display()
df.filter('country="UK"').display()
#df.select('name').display()

# 3. UPDATE → Modify Data

### Purpose:

Update existing data

In [0]:
from pyspark.sql.functions import when
df=df.withColumn('country',when(df.customer_id==1,'India').otherwise(df.country))
df.display()

In [0]:
from pyspark.sql.functions import lit
df=df.withColumn('source',lit('manual'))
df.display()

### Important:

- No direct UPDATE like SQL
- You recreate the column

### Key Points:

- Always assign back to `df`
- Logic is applied row-wise

### Practice:

- Update country for one customer
- Update multiple customers

In [0]:
from pyspark.sql.functions import when
df=df.withColumn('country',when(df.customer_id==3,'India').otherwise(df.country))
df.display()

# 4. DELETE → Remove Data

### Purpose:

Delete rows

In [0]:
df=df.filter(df.customer_id!=1)
display(df)

> Can remove almost all data accidentally

### Key Points:

- DELETE = filtering out rows
- No direct DELETE command

### Practice:

- Delete one customer
- Delete customers from UK

In [0]:
df=df.filter(df.country=='UK')

# 5. RENAME COLUMNS

Use `withColumnRenamed()` when you want to change the name of an existing column. It returns a new DataFrame.


In [0]:
import pyspark.sql.functions as F
# df= df.withColumn('Signup Year',F.year(df.signup_date))
# df.display()
df=df.withColumnRenamed('Signup Year','signup_year')
df.display()


In [0]:
df=df.withColumnRenamed('name','cust_name')
df.show()

### Practice

- Rename `customer_name` back to `name`.


In [0]:
df=df.withColumnRenamed('cust_name','name')
df.show()

# 6. DROP COLUMNS

Use `drop()` when a column is no longer required. You can drop one or multiple columns.


In [0]:
df=df.drop('signup_year')
df.display()

### Practice

- Drop the `country` column.


# 7. SORT / ORDER BY

Use `orderBy()` to sort rows. By default, sorting is ascending. Use `desc()` for descending order.


In [0]:
df=spark.read.table('reeva_dataplatform.landing.students')
df.display()

In [0]:
df.orderBy(F.desc('name')).display()

### Practice

- Sort customers by `customer_id` in ascending order.


In [0]:
df.orderBy("student_id").display()

# 8. DISTINCT

Use `distinct()` to remove duplicate rows. You can also select a column first and find its unique values.


In [0]:
df.select('city').distinct().display()

### Practice

- Show the unique customer countries.


# 9. NULL HANDLING

Use `dropna()` to remove rows containing NULL values and `fillna()` to replace NULL values.


### Practice

- Replace NULL values in `name` with `"Unknown"`.


In [0]:
from pyspark.sql.functions import when
df=spark.read.table('reeva_dataplatform.landing.students')

df=df.withColumn('country',when(df.country=='nan','Unknown').otherwise(df.country))
df=df.withColumn('city',when(df.city=='nan','NULL').otherwise(df.city))
display(df)

# 10. GROUP BY AND AGGREGATIONS

Use `groupBy()` with functions such as `count()`, `sum()`, `avg()`, `min()` and `max()` to calculate summaries by group.


In [0]:
import pyspark.sql.functions as F
df.groupBy("city").agg(F.count('student_id')).display()


### Practice

- Count how many customers are in each country.


In [0]:
df.groupBy('country').agg(F.count('student_id')).display()

# 11. JOINS

`join()` combines two DataFrames using a matching key. Common types are `inner`, `left`, `right` and `full`.


In [0]:
df_attendance=spark.read.table('reeva_dataplatform.landing.attendance')
df_attendance.display()

In [0]:
joined_df=df.join(df_attendance,'student_id','left')
joined_df.display()

### Practice

- Perform an inner join between `df` and `orders` using `customer_id`.


# 12. UNION

Use `union()` or `unionByName()` to combine rows from two DataFrames. `unionByName()` matches columns by name and is generally safer when column order may differ.


### Practice

- Create a DataFrame with one new customer and combine it with `df` using `unionByName()`.


# 13. CAST DATA TYPES

Use `cast()` to change a column's data type. This is common when reading raw data.


In [0]:
import pyspark.sql.functions as F
# df=df.withColumn('student_id',F.col('student_id').cast('int'))
df.orderBy(F.desc("student_id")).display()

### Practice

- Cast `customer_id` to `string` and inspect the schema.


In [0]:
df=df.withColumn('student_id',F.col('student_id').cast('string'))
df.display()

# 14. STRING OPERATIONS

PySpark provides functions for common string transformations such as `upper()`, `lower()`, `trim()`, `length()` and `concat_ws()`.


### Practice

- Create a new column containing the customer name in uppercase.


# 15. DATE OPERATIONS

Date functions are commonly used to extract year/month/day and calculate date differences.


### Practice

- Create a `signup_year` column from `signup_date`.


# 16. WINDOW FUNCTIONS

Window functions calculate values across related rows without collapsing them like `groupBy()`. They are useful for ranking, latest-record logic, `lag()` and `lead()`.


### Practice

- Create a row number for customers within each country ordered by `signup_date` descending.


# 17. TEMPORARY VIEW

A temporary view lets you query a DataFrame using SQL. It is session-scoped and is not stored as a permanent Unity Catalog object.


### Practice

- Create a temporary view called `uk_customers` containing only UK customers, then query it with SQL.


# 18. READ DATA

Spark can read common formats such as CSV, Parquet and Delta. In Databricks, Delta tables are especially common.


In [0]:
# Examples
# csv_df = spark.read.option("header", True).option("inferSchema", True).csv("/path/customers.csv")
# parquet_df = spark.read.parquet("/path/customers")
# delta_df = spark.read.table("catalog.schema.table_name")


### Practice

- Write the PySpark command you would use to read a Delta table named `catalog.schema.customers`.


# 19. WRITE DATA

Use `write` to persist a DataFrame. Common modes are `overwrite`, `append`, `ignore` and `error`.


In [0]:
# Example - write a DataFrame as a Delta table
# df.write.format("delta").mode("overwrite").saveAsTable("catalog.schema.customers")


### Practice

- Write `df` to a Delta table using append mode.


# 20. INSPECT A DATAFRAME

These commands are useful when developing and debugging PySpark transformations.


In [0]:
df.show()
df.printSchema()
print(df.columns)
print(df.dtypes)
print(df.count())
df.describe().show()


### Practice

- Display the schema, columns and row count for `df`.
